# 01 — Cohort & Exploratory Data Analysis

Run `python src/pipeline.py` first — this notebook reads its output from `data/processed/`.

**Note on sample size:** the MIMIC-IV *Demo* is only 100 patients. Treat everything below as a
pipeline/methodology demonstration, not a statistically powered finding — that distinction is called
out explicitly wherever it matters, rather than overclaiming what a 100-patient cohort can show.
If full MIMIC-IV credentialed access has been approved by the time this runs, point
`src/config.py`'s `DATA_DIR` at the full extract instead and rerun `pipeline.py` for a properly
powered version of every result in this notebook.

In [ ]:
import sys
sys.path.append("../src")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from config import PROCESSED_DIR

raw = pd.read_csv(PROCESSED_DIR / "feature_matrix_raw.csv")
print(raw.shape)
raw.head()

## Outcome prevalence

In-hospital mortality is a rare/imbalanced outcome — this is why AUROC/AUPRC are used as the
headline metrics in `02_Modeling.ipynb` instead of accuracy.

In [ ]:
prevalence = raw["hospital_expire_flag"].value_counts(normalize=True)
print(prevalence)
sns.countplot(x="hospital_expire_flag", data=raw)
plt.title("In-hospital mortality (0 = survived, 1 = died)")
plt.show()

## Cohort demographics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(raw["age"], bins=20, ax=axes[0])
axes[0].set_title("Age distribution")

raw["gender"].value_counts().plot.bar(ax=axes[1])
axes[1].set_title("Gender")

raw["race"].value_counts().plot.barh(ax=axes[2])
axes[2].set_title("Race/ethnicity")
plt.tight_layout()
plt.show()

## Missingness by vital/lab

A missing lab or vital in the first-24h window is itself informative (a test that wasn't ordered
because a clinician didn't think it was needed), which is why every aggregated feature has a paired
`_missing` indicator (see `src/features.py`) rather than being silently imputed away here.

In [ ]:
missing_cols = [c for c in raw.columns if c.endswith("_missing")]
missing_rate = raw[missing_cols].mean().sort_values(ascending=False)

plt.figure(figsize=(8, 6))
missing_rate.plot.barh()
plt.xlabel("Fraction of stays missing this feature in the first 24h")
plt.tight_layout()
plt.show()

missing_rate

## Comorbidity burden vs. outcome

In [ ]:
sns.boxplot(x="hospital_expire_flag", y="comorbidity_score", data=raw)
plt.title("Comorbidity score by outcome")
plt.show()